# MultiIndex, COVID, and Pollution — Solved (Task 21)

This solutions notebook revisits the **MultiIndex** and applies it to two substantial real-world problems. It begins with a focused puzzle about swapping levels between the index and the columns, then moves to the COVID-19 global time series — where a wide, one-column-per-day layout must be melted, merged, and re-indexed by country and province — and closes with the world's most-air-polluted-cities dataset, where hierarchical indexing makes country/city queries and charts natural.

Q-1 is deliberately abstract: you are given a DataFrame whose rows are `(branch, year)` tuples and whose columns are `(city, metric)` pairs, and you must transform it so that `('cse','ece')` becomes the top level of the **columns** and `('delhi','mumbai')` becomes the top level of the **index**, then sort the row index. The elegant solution chains `unstack(level=0).stack(level=0).swaplevel()` with `swaplevel(axis=1)` and `sort_index()` — a masterclass in moving levels around with `stack`, `unstack`, and `swaplevel`. Understanding *which* level ends up where is the whole exercise.

Q-2 assembles the COVID data. Two published CSVs (confirmed cases and deaths) each hold one row per location and one **column per date**. You melt both into tidy `(location, date, value)` form, merge them on the shared location-and-date keys, and select `date`, `num_cases`, and `num_deaths`. The resulting frame carries a `Country/Region → Province/State` MultiIndex — the requested structure. Q-3 then groups by country to compute the death percentage (deaths ÷ cases × 100), revealing outliers and a data-quality curiosity (North Korea's >100% figure points to an undercounted denominator). Q-4 drills into India, deriving a `new_cases` column as the day-over-day difference using the `shift` function, with the first day left as `NaN`.

Q-5 works the air-pollution dataset: after dropping a stray `Unnamed: 0` column and setting a `Country`/`City/Town` index, you find India's most and least polluted cities by `PM10`, repeat for China, plot Poland's cities as a pie chart, bar-chart Israel and Qatar on `PM2.5` and `PM10`, and finally collapse the whole MultiIndex frame to a Series with `stack()` without losing any information. Each answer is a one-liner made possible by the hierarchical index.

## 0. Setup

Import `pandas` to begin.

In [1]:
import pandas as pd

# 1. Swapping Index and Column Levels

A two-level index and a two-level column header can be re-arranged almost arbitrarily with `unstack`, `stack`, and `swaplevel`. This puzzle asks for one particular arrangement.

###`Q-1:` You are given a Multi index DataFrame. You task is to swap level-0 index with level-0 column.
* Change Branch -> ('cse', 'ece') as columns at level-0 and ('delhi'	'mumbai') as level-0 Index
* And Sort on row index level-0

In [2]:
# Given Code Snippets
index_val = [('cse',2019),('cse',2020),('cse',2021),('cse',2022),('ece',2019),('ece',2020),('ece',2021),('ece',2022)]
multiindex = pd.MultiIndex.from_tuples(index_val)
df = pd.DataFrame(
    [
        [1,2,0,0],
        [3,4,0,0],
        [5,6,0,0],
        [7,8,0,0],
        [9,10,0,0],
        [11,12,0,0],
        [13,14,0,0],
        [15,16,0,0],
    ],
    index = multiindex,
    columns = pd.MultiIndex.from_product([['delhi','mumbai'],['avg_package','students']])
)

df

               delhi               mumbai         
         avg_package students avg_package students
cse 2019           1        2           0        0
    2020           3        4           0        0
    2021           5        6           0        0
    2022           7        8           0        0
ece 2019           9       10           0        0
    2020          11       12           0        0
    2021          13       14           0        0
    2022          15       16           0        0

## 1.1 Approach — unstack, stack, swaplevel, sort

The given frame `df` has rows labelled `(cse, 2019) ... (ece, 2022)` and columns labelled `(delhi, avg_package)`, `(delhi, students)`, `(mumbai, avg_package)`, `(mumbai, students)`. The goal is to move the branch level from the rows to the outer level of the columns, and the city level from the columns to the rows.

The solution is `df.unstack(level=0).stack(level=0).swaplevel().swaplevel(axis=1).sort_index()`. Reading it step by step: `unstack(level=0)` pushes the outer index level (the branch) out into the columns; `stack(level=0)` folds part of the result back into the index; `swaplevel()` (no axis, so rows) reorders the row levels; `swaplevel(axis=1)` reorders the **column** levels; and `sort_index()` sorts rows by level 0. The printed result has columns headed `cse / ece` on top and `avg_package / students` beneath, with rows `delhi/cse...` grouped by city and year. The exact chaining is less important than the mental model: `stack` and `unstack` shuttle a level between index and columns, `swaplevel` reorders within an axis, and `sort_index` brings order. Practise predicting the shape after each call — that skill transfers directly to the `pivot_table` reshaping questions in later notebooks.

In [9]:
# code here
df.unstack(level=0).stack(level=0).swaplevel().swaplevel(axis=1).sort_index()

                    cse         ece      cse      ece
            avg_package avg_package students students
delhi  2019           1           9        2       10
       2020           3          11        4       12
       2021           5          13        6       14
       2022           7          15        8       16
mumbai 2019           0           0        0        0
       2020           0           0        0        0
       2021           0           0        0        0
       2022           0           0        0        0

# 2. COVID-19 Time Series

Two global datasets — confirmed cases and deaths — each stored in a wide, one-column-per-day format. The tasks reshape them, build a MultiIndex, answer a death-rate question, and compute India's daily new cases.

###`Q-2:` Covid Cases Data Set Problem
 Make a DataFrame Using both Covid Dataset. With Country name as Level-0 and Provinance/State as Level-1 Index and date, No of cases and No of Deaths as Columns.

## 2.1 Loading the COVID files

`pd.read_csv` reads the two published Google Sheets URLs into `confirm` and `deaths`. Both share the same layout: identifier columns `Province/State`, `Country/Region`, `Lat`, and `Long`, followed by **one column per date** from `1/22/20` onward, holding cumulative totals. Inspecting `deaths.head(2)` confirms the shape and the many date columns. This is 'wide' data, and the cumulative nature matters: each value is the running total, not a daily increment, so any per-day analysis must difference consecutive dates (exactly what Q-4 does). Also note that `Province/State` is often `NaN` for countries without provinces — that missing key will still participate in the merge and index, which is fine but worth remembering when filtering.

In [28]:
confirm = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vTmqT3kxd0i0RUFiEnwA1Hboiunv28MeNTatZsIEqlPPB7mHrl0ttJL7utZ23_1s5FW8ZjODmB8jHIi/pub?gid=2142019845&single=true&output=csv')

deaths = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vQn4nLCKIVZMw4e89YeRqgKaSAAwRFaZ6ILMW_FUejZ2tkv3Np1f9gD4QOl3ASHeLzWjOjfmLDKcCOF/pub?gid=212966958&single=true&output=csv")

In [29]:
deaths.head(2)

  Province/State Country/Region       Lat       Long  1/22/20  1/23/20  \
0            NaN    Afghanistan  33.93911  67.709953        0        0   
1            NaN        Albania  41.15330  20.168300        0        0   

   1/24/20  1/25/20  1/26/20  1/27/20  ...  12/24/22  12/25/22  12/26/22  \
0        0        0        0        0  ...      7845      7846      7846   
1        0        0        0        0  ...      3595      3595      3595   

   12/27/22  12/28/22  12/29/22  12/30/22  12/31/22  1/1/23  1/2/23  
0      7846      7846      7847      7847      7849    7849    7849  
1      3595      3595      3595      3595      3595    3595    3595  

[2 rows x 1081 columns]

## 2.2 Melt, merge, and set the MultiIndex

To convert the wide tables into tidy form, `melt` is applied to each:
```python
deaths = deaths.melt(id_vars=['Province/State','Country/Region','Lat','Long'],
                     var_name='date', value_name='num_deaths')
confirm = confirm.melt(id_vars=['Province/State','Country/Region','Lat','Long'],
                       var_name='date', value_name='num_cases')
```
`id_vars` keeps the four location columns fixed; every date column collapses into rows of `(location, date, value)`. The two tidy frames are then joined on all five shared keys — `['Province/State','Country/Region','Lat','Long','date']` — so each `(location, date)` row carries both a case total and a death total. Selecting `['date','num_cases','num_deaths']` gives the requested columns, and the `Country/Region → Province/State` MultiIndex is established from the location keys. This melt-merge-index pipeline is the canonical way to combine several wide COVID-style files; a merge on the full key set ensures no location is dropped or duplicated.

In [30]:
deaths = deaths.melt(id_vars=['Province/State','Country/Region','Lat','Long'],var_name='date',value_name='num_deaths')
confirm = confirm.melt(id_vars=['Province/State','Country/Region','Lat','Long'],var_name='date',value_name='num_cases')

In [31]:
final_df = confirm.merge(deaths,on=['Province/State','Country/Region','Lat','Long','date'])
final_df

       Province/State        Country/Region        Lat        Long     date  \
0                 NaN           Afghanistan  33.939110   67.709953  1/22/20   
1                 NaN               Albania  41.153300   20.168300  1/22/20   
2                 NaN               Algeria  28.033900    1.659600  1/22/20   
3                 NaN               Andorra  42.506300    1.521800  1/22/20   
4                 NaN                Angola -11.202700   17.873900  1/22/20   
...               ...                   ...        ...         ...      ...   
311248            NaN    West Bank and Gaza  31.952200   35.233200   1/2/23   
311249            NaN  Winter Olympics 2022  39.904200  116.407400   1/2/23   
311250            NaN                 Yemen  15.552727   48.516388   1/2/23   
311251            NaN                Zambia -13.133897   27.849332   1/2/23   
311252            NaN              Zimbabwe -19.015438   29.154857   1/2/23   

        num_cases  num_deaths  
0               0  

In [34]:
#final_df.set_index(['Country/Region','Province/State'],inplace=True)
final_df = final_df[['date','num_cases','num_deaths']]
final_df

                                        date  num_cases  num_deaths
Country/Region       Province/State                                
Afghanistan          NaN             1/22/20          0           0
Albania              NaN             1/22/20          0           0
Algeria              NaN             1/22/20          0           0
Andorra              NaN             1/22/20          0           0
Angola               NaN             1/22/20          0           0
...                                      ...        ...         ...
West Bank and Gaza   NaN              1/2/23     703228        5708
Winter Olympics 2022 NaN              1/2/23        535           0
Yemen                NaN              1/2/23      11945        2159
Zambia               NaN              1/2/23     334661        4024
Zimbabwe             NaN              1/2/23     259981        5637

[311253 rows x 3 columns]

# Task Q-3 — Highest Death Percentage

**Goal:** Find the country (or region) with the highest death percentage — total deaths ÷ total cases × 100.

###`Q-3:` Show Country with Highest death percent out of confirmed Cases. 

## 2.3 Approach — group by country, divide, sort

`final_df.groupby('Country/Region').sum()` collapses the per-province, per-date rows into country totals for `num_cases` and `num_deaths` (numeric columns only). Then `temp_df['death_percentage'] = temp_df['num_deaths'] / temp_df['num_cases'] * 100` computes the ratio, and `sort_values('death_percentage', ascending=False)` ranks it.

The result contains a striking anomaly: **Korea, North shows 600%** — deaths exceed cases, which is impossible epidemiologically. That is a data-quality flag, not a real finding: the reported death figure for that region is inconsistent with the case count, likely due to reporting corrections or misalignment in the source. MS Zaandam (a cruise ship) at 22.2% and Yemen at 19.4% are plausible high-mortality contexts (limited healthcare, testing gaps). The lesson is twofold: division exposes data errors like this vividly, and any 'highest ratio' ranking should be inspected for small or corrupted denominators before it is believed. A careful analyst would filter out regions with implausibly few cases or flag the >100% rows explicitly.

In [58]:
# code here
temp_df = final_df.groupby('Country/Region').sum()
temp_df['death_percentage'] = (temp_df['num_deaths']/temp_df['num_cases'])*100
temp_df.sort_values('death_percentage',ascending=False)

                       num_cases  num_deaths  death_percentage
Country/Region                                                
Korea, North                 234        1404        600.000000
MS Zaandam                  9071        2014         22.202624
Yemen                    7091065     1372952         19.361718
Sudan                   38729339     2850576          7.360250
Peru                  2203724423   156296296          7.092370
...                          ...         ...               ...
Holy See                   24893           0          0.000000
Summer Olympics 2020      441598           0          0.000000
Winter Olympics 2022      179152           0          0.000000
Tuvalu                    137771           0          0.000000
Antarctica                  4235           0          0.000000

[201 rows x 3 columns]

# Task Q-4 — India's Daily New Cases

**Goal:** Build a DataFrame for India from the COVID data with an extra `new_cases` column equal to the day-over-day change in cumulative cases (first day `NaN`). Use `shift`.

### `Q-4` : Make a dataframe for India from Covid Data with one extra column representing no of new cases. 

* Just for Assumption "No of new cases" will be equal to difference of "no of cases" with previous day.
* First day new cases will be NaN or equal to no of cases

Say on 12/30/22 No of cases is - 44679608	and a day previous (12/29/22) no of cases is - 44679382.

Then for 12/30/22 -> No of New Cases  = 44679608 - 44679382 = 

Note:- Try using shift Function

Try using the shift function

```
s = pd.Series([1,2,3,4,5,6])
s #-> [1,2,3,4,5,6]
s.shift(1) #-> [NaN, 1,2,3,4,5]
s.shift(-1) #-> [2,3,4,5,6, NaN]
```

## 2.4 Approach — select, difference with shift, keep order

`india_df = final_df.loc['India'].copy()` slices out India using the level-0 country key and copies it so edits do not warn. Because the rows are already ordered by date (the melt preserved the column order), the day-over-day difference is simply:
```python
india_df['new_cases'] = india_df['num_cases'] - india_df['num_cases'].shift(1)
```
`shift(1)` moves the cumulative series down by one row, so subtracting it from the original yields, for each day, `cases_today − cases_yesterday`. The first row has no predecessor, so `shift` produces `NaN` there — matching the note that the first day's new cases are `NaN` (or could be set equal to the total). The hint cell illustrates exactly this behaviour on `[1,2,3,4,5,6]`: `shift(1)` gives `[NaN,1,2,3,4,5]` and `shift(-1)` gives `[2,3,4,5,6,NaN]` for the reverse difference.

The critical assumption is that the rows are **sorted by date**; differencing unsorted cumulative data produces nonsense. In this dataset the melt preserved the original date-column order, so it holds — but on real data you would sort explicitly first (`india_df = india_df.sort_index()` or by a parsed date). `diff()` is the convenience alias for `series - series.shift(1)`, and the pattern generalises to any cumulative quantity: revenue, signups, page views.

In [65]:
# code here
india_df = final_df.loc['India'].copy()
india_df['new_cases'] = india_df['num_cases'] - india_df['num_cases'].shift(1)
india_df

                    date  num_cases  num_deaths  new_cases
Province/State                                            
NaN              1/22/20          0           0        NaN
NaN              1/23/20          0           0        0.0
NaN              1/24/20          0           0        0.0
NaN              1/25/20          0           0        0.0
NaN              1/26/20          0           0        0.0
...                  ...        ...         ...        ...
NaN             12/29/22   44679382      530699      243.0
NaN             12/30/22   44679608      530702      226.0
NaN             12/31/22   44679873      530705      265.0
NaN               1/1/23   44680046      530707      173.0
NaN               1/2/23   44680180      530707      134.0

[1077 rows x 4 columns]

# 3. World's Most Air-Polluted Cities

A flat CSV is transformed into a `Country`/`City/Town` MultiIndex frame, then queried and plotted with `.loc` slicing.

### `Q-5:` Read the Dataset using the below given link and create a multi-index dataframe using the columns "Country" and "City/Town". 

This dataset is about the most polluted cities in the world. You can get details from [here](https://www.kaggle.com/datasets/rajkumarpandey02/worlds-most-air-polluted-countries-cities).

**Dataset link:** https://tinyurl.com/2fe6vz4u Directly use this link to read.

**Task:**
1. Find out the name of the city of India which is most poluted based on PM10.
2. Find out the name of the city of India which has minumum pollution level based on PM10.
3. Do same operations (like 1 and 2) with the country China.
4. Make a pie chart based on the column "PM10" of the country Poland.
5. Make a bar chart based on the columns "PM2.5" and "PM10" of the countries Israel and Qatar.
6. Convert this MultiIndex DataFrame to Series by retaining the informations.

## 3.1 Loading and reshaping the pollution data

The tinyurl link is read with `pd.read_csv`, and `df.head()` shows columns `Unnamed: 0`, `Position`, `Country`, `City/Town`, `Year`, `PM2.5`, `Temporal coverage`, `PM10`, and others. The `Unnamed: 0` column is the classic artefact of a CSV written with its index — it carries no information, so `df.drop(columns=['Unnamed: 0'], inplace=True)` removes it. Then `df.set_index(['Country','City/Town'], inplace=True)` promotes the two natural keys into a **MultiIndex**, which is the structure every subsequent task assumes.

With that index, the frame answers questions by `df.loc['India']` returning all Indian cities and `df.loc[['Israel','Qatar']]` returning a subset of countries. Levels can be sliced by outer or inner key, and `.loc` on a MultiIndex accepts a single country name or a list. The `Position` column remains as a within-country ranking, while `PM2.5`, `PM10`, and `Temporal coverage` are the measurements. Setting the index *after* dropping the junk column matters only cosmetically here, but it keeps the MultiIndex clean. This three-step preparation — read, drop junk, set index — is the standard opening for any country/city grouped analysis.

In [74]:
# code here
df = pd.read_csv('https://tinyurl.com/2fe6vz4u')
df.head()

   Unnamed: 0  Position Country  City/Town  Year  PM2.5 Temporal coverage  \
0           0         1   India     Kanpur  2016    173              >75%   
1           1         2   India  Faridabad  2016    172              >75%   
2           2         3   India       Gaya  2016    149        50% -< 75%   
3           3         4   India   Varanasi  2016    146              >75%   
4           4         5   India      Patna  2016    144              >75%   

   PM10 Temporal coverage.1  Database version (year)  
0   319                 NaN                     2018  
1   316                 NaN                     2018  
2   275                 NaN                     2018  
3   260                 NaN                     2018  
4   266                 NaN                     2018  

In [75]:
df.drop(columns=['Unnamed: 0'],inplace=True)

In [76]:
df.head()

   Position Country  City/Town  Year  PM2.5 Temporal coverage  PM10  \
0         1   India     Kanpur  2016    173              >75%   319   
1         2   India  Faridabad  2016    172              >75%   316   
2         3   India       Gaya  2016    149        50% -< 75%   275   
3         4   India   Varanasi  2016    146              >75%   260   
4         5   India      Patna  2016    144              >75%   266   

  Temporal coverage.1  Database version (year)  
0                 NaN                     2018  
1                 NaN                     2018  
2                 NaN                     2018  
3                 NaN                     2018  
4                 NaN                     2018  

In [78]:
df.set_index(['Country','City/Town'],inplace=True)

In [79]:
df.head()

                   Position  Year  PM2.5 Temporal coverage  PM10  \
Country City/Town                                                  
India   Kanpur            1  2016    173              >75%   319   
        Faridabad         2  2016    172              >75%   316   
        Gaya              3  2016    149        50% -< 75%   275   
        Varanasi          4  2016    146              >75%   260   
        Patna             5  2016    144              >75%   266   

                  Temporal coverage.1  Database version (year)  
Country City/Town                                               
India   Kanpur                    NaN                     2018  
        Faridabad                 NaN                     2018  
        Gaya                      NaN                     2018  
        Varanasi                  NaN                     2018  
        Patna                     NaN                     2018  

## 3.2 Queries, charts, and collapsing to a Series

**Most/least polluted cities.** `df.loc['India'].sort_values('PM10', ascending=False).head(1).index[0]` returns `'Kanpur'` (India's worst PM10); the `.tail(1)` version returns `'Aizawl'` (the least polluted Indian city in this data). The same pattern for China gives `'Jinan'` as most polluted and `'Anshun'` as least. The recipe — slice a country, sort by a pollutant column, take the first or last row, read the index — is flexible and works because `City/Town` (the inner level) remains the row index after `.loc['India']`.

**Charts.** `df.loc['Poland']['PM10'].plot(kind='pie', autopct='%0.1f%%')` turns Poland's city PM10 values into a pie chart, with `autopct` labelling each slice's percentage — showing how much of Poland's total particulate load each city contributes. `df.loc[['Israel','Qatar']][['PM2.5','PM10']].plot(kind='bar')` groups the two pollutants side by side for both countries. Before plotting, make sure the values are numeric and non-negative, since pie charts especially require positive slices.

**Collapse to a Series.** `df.stack()` folds the column axis into the row index, turning every measurement into a `(Country, City, metric) → value` entry — a long MultiIndex Series that retains *all* information but in one dimension. The reverse (`unstack`) would widen it again. This is the same Series↔DataFrame duality seen with `stack`/`unstack` throughout the MultiIndex sessions, and it is how you hand a rich hierarchical frame to code that expects a single labelled column.

In [83]:
df.loc['India'].sort_values('PM10',ascending=False).head(1).index[0]

'Kanpur'

In [84]:
df.loc['India'].sort_values('PM10',ascending=False).tail(1).index[0]

'Aizawl'

In [85]:
df.loc['China'].sort_values('PM10',ascending=False).tail(1).index[0]

'Anshun'

In [86]:
df.loc['China'].sort_values('PM10',ascending=False).head(1).index[0]

'Jinan'

In [89]:
df.loc['Poland']['PM10'].plot(kind='pie',autopct="%0.1f%%")

<Figure size 432x288 with 1 Axes>

In [92]:
df.loc[['Israel','Qatar']][['PM2.5','PM10']].plot(kind='bar')

<Figure size 432x288 with 1 Axes>

In [94]:
df.stack()

Country  City/Town                         
India    Kanpur     Position                      1
                    Year                       2016
                    PM2.5                       173
                    Temporal coverage          >75%
                    PM10                        319
                                               ... 
Turkey   Sivas      PM2.5                        27
                    Temporal coverage          >75%
                    PM10                         56
                    Temporal coverage.1        >75%
                    Database version (year)    2018
Length: 2638, dtype: object

## Summary

Task 21 put the MultiIndex through a level-swapping puzzle and two real datasets. Q-1 re-arranged a nested frame by chaining `unstack(level=0)`, `stack(level=0)`, `swaplevel()`, `swaplevel(axis=1)`, and `sort_index()` — moving the branch to the column header and the city to the row index. On COVID data you loaded two wide, one-column-per-day CSVs, melted both with `id_vars`, merged on the full location-and-date key set, and selected `date`, `num_cases`, and `num_deaths` under a `Country/Region → Province/State` index; grouped by country to rank death percentages (spotting the >100% data anomaly); and derived India's `new_cases` as `num_cases − num_cases.shift(1)` with the first day left `NaN`. On the pollution data you dropped the junk `Unnamed: 0` column, set a `Country`/`City/Town` index, sliced countries with `.loc` to find India's and China's most/least polluted cities (Kanpur/Jinan vs Aizawl/Anshun), plotted a Poland pie and an Israel/Qatar bar chart, and collapsed the frame to a Series with `stack()`. Core themes: move levels deliberately, melt wide time series, and let the index do the filtering.